# Plan B - Databricks Personal Connections Report (Version 7)

Scans the entire Power BI / Fabric tenant using the Scanner API and produces
a report of all Databricks datasources with owner contact details.

**Version 7 changes vs Version 6:**
- Runs entirely in Fabric Spark — no interactive `display()` calls
- Results are written to **Lakehouse Delta tables** (queryable via SQL endpoint)
- Diagnostic cell removed — safe for scheduled execution inside a Fabric Pipeline

**Output tables written to the default Lakehouse:**
- `databricks_all_connections` — every Databricks datasource found in the tenant
- `databricks_personal_connections` — subset that is NOT yet on the VNet Gateway

In [ ]:
import json
import time
import pandas as pd
import requests as http_requests

# Spark session is available automatically in Fabric notebooks
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

try:
    pbi_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
except Exception:
    pbi_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com/")

HEADERS = {
    "Authorization": f"Bearer {pbi_token}",
    "Content-Type":  "application/json",
}

PBI_API = "https://api.powerbi.com/v1.0/myorg"

# ── Known VNet Gateway IDs ── update this set with your tenant's VNet GW IDs
KNOWN_VNET_GATEWAY_IDS = {
    "3e481240-199c-402f-8fe3-ae5ef09b1ab4",
}

# ── Lakehouse table names (written to the default attached Lakehouse) ──
TABLE_ALL      = "databricks_all_connections"
TABLE_PERSONAL = "databricks_personal_connections"

print("Setup complete.")

In [ ]:
# Fetch capacities
print("Fetching capacities...")
cap_map = {}

try:
    cap_resp = http_requests.get(f"{PBI_API}/admin/capacities", headers=HEADERS)
    if cap_resp.status_code == 200:
        for cap in cap_resp.json().get("value", []):
            cap_map[cap["id"]] = {
                "capacity_name": cap.get("displayName", ""),
                "capacity_sku":  cap.get("sku", ""),
            }
        print(f"{len(cap_map)} capacities loaded.")
    else:
        print(f"Status {cap_resp.status_code}: {cap_resp.text[:300]}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Fetch all workspaces
print("Fetching all workspaces...")
all_workspaces = []
top = 5000
skip = 0

while True:
    ws_resp = http_requests.get(
        f"{PBI_API}/admin/groups?$top={top}&$skip={skip}",
        headers=HEADERS,
    )
    if ws_resp.status_code == 429:
        retry = int(ws_resp.headers.get("Retry-After", 30))
        print(f"Rate limited. Sleeping {retry}s...")
        time.sleep(retry)
        continue
    if ws_resp.status_code != 200:
        print(f"Status {ws_resp.status_code}: {ws_resp.text[:300]}")
        break
    batch = ws_resp.json().get("value", [])
    if not batch:
        break
    all_workspaces.extend(batch)
    if len(batch) < top:
        break
    skip += top

ws_map = {}
for ws in all_workspaces:
    ws_map[ws["id"]] = {
        "workspace_name": ws.get("name", ""),
        "capacity_id":    ws.get("capacityId", ""),
    }

print(f"Total workspaces: {len(all_workspaces)}")

In [ ]:
# Batch scan using Scanner API
BATCH_SIZE = 100
POLL_INTERVAL = 5
MAX_POLL_TIME = 300

ws_ids = [ws["id"] for ws in all_workspaces]
total_batches = (len(ws_ids) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Scanning {len(ws_ids)} workspaces in {total_batches} batches...")

all_scan_workspaces = []
scan_errors = []

SCAN_URL = (
    f"{PBI_API}/admin/workspaces/getInfo"
    f"?datasourceDetails=True"
    f"&lineage=True"
    f"&datasetSchema=False"
    f"&datasetExpressions=False"
)

for batch_num in range(total_batches):
    start = batch_num * BATCH_SIZE
    end   = start + BATCH_SIZE
    batch_ids = ws_ids[start:end]

    print(f"  Batch {batch_num + 1}/{total_batches} ({len(batch_ids)} workspaces)...")

    scan_body = {"workspaces": batch_ids}

    try:
        scan_resp = http_requests.post(SCAN_URL, headers=HEADERS, json=scan_body)
        if scan_resp.status_code == 429:
            retry = int(scan_resp.headers.get("Retry-After", 60))
            print(f"    Rate limited. Sleeping {retry}s...")
            time.sleep(retry)
            scan_resp = http_requests.post(SCAN_URL, headers=HEADERS, json=scan_body)
        if scan_resp.status_code not in (200, 202):
            print(f"    Failed: {scan_resp.status_code} {scan_resp.text[:300]}")
            scan_errors.append({"batch": batch_num, "error": scan_resp.text[:500]})
            continue
        scan_info = scan_resp.json()
        scan_id = scan_info.get("id", scan_info.get("scanId", ""))
    except Exception as e:
        print(f"    Error: {e}")
        scan_errors.append({"batch": batch_num, "error": str(e)})
        continue

    elapsed = 0
    scan_status = ""
    while elapsed < MAX_POLL_TIME:
        time.sleep(POLL_INTERVAL)
        elapsed += POLL_INTERVAL
        try:
            status_resp = http_requests.get(
                f"{PBI_API}/admin/workspaces/scanStatus/{scan_id}",
                headers=HEADERS,
            )
            if status_resp.status_code == 200:
                scan_status = status_resp.json().get("status", "").lower()
                if scan_status == "succeeded":
                    break
                elif scan_status == "failed":
                    break
            elif status_resp.status_code == 429:
                retry = int(status_resp.headers.get("Retry-After", 15))
                time.sleep(retry)
                elapsed += retry
        except Exception:
            pass

    if scan_status != "succeeded":
        print(f"    Not succeeded (status: {scan_status})")
        scan_errors.append({"batch": batch_num, "status": scan_status})
        continue

    try:
        result_resp = http_requests.get(
            f"{PBI_API}/admin/workspaces/scanResult/{scan_id}",
            headers=HEADERS,
        )
        if result_resp.status_code == 200:
            ws_results = result_resp.json().get("workspaces", [])
            all_scan_workspaces.extend(ws_results)
            print(f"    OK: {len(ws_results)} workspaces.")
        else:
            print(f"    Result failed: {result_resp.status_code}")
    except Exception as e:
        print(f"    Error: {e}")

    time.sleep(2)

print(f"\nScan complete. {len(all_scan_workspaces)} workspaces scanned.")
if scan_errors:
    print(f"{len(scan_errors)} batch errors.")

In [ ]:
# Parse results — tries ALL known locations for datasource info
print("Parsing scan results...\n")
results = []

for ws_data in all_scan_workspaces:
    ws_id   = ws_data.get("id", "")
    ws_name = ws_data.get("name", "")

    capacity_id = ws_data.get("capacityId", "")
    if not capacity_id:
        capacity_id = ws_map.get(ws_id, {}).get("capacity_id", "")
    cap_info      = cap_map.get(capacity_id, {})
    capacity_name = cap_info.get("capacity_name", "")
    capacity_sku  = cap_info.get("capacity_sku", "")

    # Build datasource instance lookup from ALL possible API response locations
    ds_instance_map = {}

    # Location 1: workspace.datasourceInstances
    for inst in ws_data.get("datasourceInstances", []):
        inst_id = inst.get("datasourceId", inst.get("datasourceInstanceId", inst.get("id", "")))
        if inst_id:
            ds_instance_map[inst_id] = inst

    # Location 2: workspace.datasets[].datasourceInstances
    for dataset in ws_data.get("datasets", []):
        for inst in dataset.get("datasourceInstances", []):
            inst_id = inst.get("datasourceId", inst.get("datasourceInstanceId", inst.get("id", "")))
            if inst_id:
                ds_instance_map[inst_id] = inst

    # Location 3: workspace.datasets[].datasources
    for dataset in ws_data.get("datasets", []):
        for inst in dataset.get("datasources", []):
            inst_id = inst.get("datasourceId", inst.get("datasourceInstanceId", inst.get("id", "")))
            if inst_id:
                ds_instance_map[inst_id] = inst

    for dataset in ws_data.get("datasets", []):
        dataset_id    = dataset.get("id", "")
        dataset_name  = dataset.get("name", "")
        configured_by = dataset.get("configuredBy", "")

        usage_ids = [
            u.get("datasourceInstanceId", "")
            for u in dataset.get("datasourceUsages", [])
        ]

        if ds_instance_map:
            for inst_id in usage_ids:
                src = ds_instance_map.get(inst_id, {})
                if not src:
                    continue

                ds_type  = src.get("datasourceType", "")
                conn_raw = src.get("connectionDetails", {})
                conn_str = json.dumps(conn_raw) if conn_raw else "{}"
                gw_id    = src.get("gatewayId", "")

                is_dbx = False
                if ds_type.lower() in ("databricks", "azuredatabricks"):
                    is_dbx = True
                elif ds_type.lower() == "extension":
                    if "databricks" in conn_str.lower():
                        is_dbx = True
                if not is_dbx:
                    continue

                if not gw_id:
                    gw_class = "No Gateway"
                elif gw_id in KNOWN_VNET_GATEWAY_IDS:
                    gw_class = "VNet Gateway"
                else:
                    gw_class = "Personal Cloud Gateway"

                host_val      = conn_raw.get("host", conn_raw.get("server", conn_raw.get("path", "")))
                http_path_val = conn_raw.get("httpPath", conn_raw.get("database", ""))
                if isinstance(host_val, str) and host_val.strip().startswith("{"):
                    try:
                        parsed = json.loads(host_val)
                        host_val = parsed.get("host", parsed.get("server", host_val))
                        if not http_path_val:
                            http_path_val = parsed.get("httpPath", "")
                    except Exception:
                        pass

                results.append({
                    "capacity_id":        capacity_id,
                    "capacity_name":      capacity_name,
                    "capacity_sku":       capacity_sku,
                    "workspace_id":       ws_id,
                    "workspace_name":     ws_name,
                    "dataset_id":         dataset_id,
                    "dataset_name":       dataset_name,
                    "datasource_id":      inst_id,
                    "datasource_type":    ds_type,
                    "gateway_id":         gw_id,
                    "gateway_class":      gw_class,
                    "host":               host_val,
                    "http_path":          http_path_val,
                    "connection_details": conn_str,
                    "owner_name":         configured_by,
                    "owner_email":        configured_by,
                })
        else:
            # No instance details available — record the IDs for enrichment
            for inst_id in usage_ids:
                results.append({
                    "capacity_id":        capacity_id,
                    "capacity_name":      capacity_name,
                    "capacity_sku":       capacity_sku,
                    "workspace_id":       ws_id,
                    "workspace_name":     ws_name,
                    "dataset_id":         dataset_id,
                    "dataset_name":       dataset_name,
                    "datasource_id":      inst_id,
                    "datasource_type":    "(scan has no details)",
                    "gateway_id":         "",
                    "gateway_class":      "(unknown)",
                    "host":               "",
                    "http_path":          "",
                    "connection_details": "",
                    "owner_name":         configured_by,
                    "owner_email":        configured_by,
                })

print(f"Total rows: {len(results)}")
print(f"Rows with details: {sum(1 for r in results if r['datasource_type'] != '(scan has no details)')}")
print(f"Rows without details: {sum(1 for r in results if r['datasource_type'] == '(scan has no details)')}")

In [ ]:
# If scan didn't return datasource details, enrich using the
# per-dataset admin API (targeted calls only, with rate limit handling)

needs_enrichment = [r for r in results if r["datasource_type"] == "(scan has no details)"]

if needs_enrichment:
    print(f"Scan didn't return datasource details.")
    print(f"Enriching {len(needs_enrichment)} datasource references via targeted API calls...")
    print(f"(Only calling datasets that have datasourceUsages, with rate limit handling)\n")

    datasets_to_enrich = {}
    for r in needs_enrichment:
        ds_id = r["dataset_id"]
        if ds_id not in datasets_to_enrich:
            datasets_to_enrich[ds_id] = r["workspace_id"]

    print(f"Unique datasets to query: {len(datasets_to_enrich)}")

    datasource_cache = {}
    api_calls = 0

    for idx, (dataset_id, workspace_id) in enumerate(datasets_to_enrich.items()):
        if (idx + 1) % 50 == 0:
            print(f"  Progress: {idx + 1}/{len(datasets_to_enrich)}")

        try:
            resp = http_requests.get(
                f"{PBI_API}/admin/datasets/{dataset_id}/datasources",
                headers=HEADERS,
            )
            api_calls += 1

            if resp.status_code == 429:
                retry = int(resp.headers.get("Retry-After", 30))
                print(f"  Rate limited at call {api_calls}. Sleeping {retry}s...")
                time.sleep(retry)
                resp = http_requests.get(
                    f"{PBI_API}/admin/datasets/{dataset_id}/datasources",
                    headers=HEADERS,
                )
                api_calls += 1

            if resp.status_code == 200:
                for src in resp.json().get("value", []):
                    src_inst_id = src.get("datasourceId", "")
                    if src_inst_id:
                        datasource_cache[src_inst_id] = src
        except Exception:
            pass

        # Throttle: ~200 calls/min
        if api_calls % 10 == 0:
            time.sleep(1)

    print(f"\nDone. {api_calls} API calls, {len(datasource_cache)} datasources cached.")

    enriched_results = []
    for r in results:
        if r["datasource_type"] == "(scan has no details)":
            src = datasource_cache.get(r["datasource_id"], {})
            if src:
                ds_type  = src.get("datasourceType", "")
                conn_raw = src.get("connectionDetails", {})
                conn_str = json.dumps(conn_raw) if conn_raw else "{}"
                gw_id    = src.get("gatewayId", "")

                is_dbx = False
                if ds_type.lower() in ("databricks", "azuredatabricks"):
                    is_dbx = True
                elif ds_type.lower() == "extension":
                    if "databricks" in conn_str.lower():
                        is_dbx = True

                if not is_dbx:
                    continue

                if not gw_id:
                    gw_class = "No Gateway"
                elif gw_id in KNOWN_VNET_GATEWAY_IDS:
                    gw_class = "VNet Gateway"
                else:
                    gw_class = "Personal Cloud Gateway"

                host_val      = conn_raw.get("host", conn_raw.get("server", conn_raw.get("path", "")))
                http_path_val = conn_raw.get("httpPath", conn_raw.get("database", ""))
                if isinstance(host_val, str) and host_val.strip().startswith("{"):
                    try:
                        parsed = json.loads(host_val)
                        host_val = parsed.get("host", parsed.get("server", host_val))
                        if not http_path_val:
                            http_path_val = parsed.get("httpPath", "")
                    except Exception:
                        pass

                r["datasource_type"]    = ds_type
                r["gateway_id"]         = gw_id
                r["gateway_class"]      = gw_class
                r["host"]               = host_val
                r["http_path"]          = http_path_val
                r["connection_details"] = conn_str
                enriched_results.append(r)
        else:
            enriched_results.append(r)

    results = enriched_results
    print(f"\nFinal Databricks rows: {len(results)}")
else:
    print("Scan returned datasource details directly. No enrichment needed.")

In [ ]:
# Build final DataFrames and write to Lakehouse Delta tables
df_planb = pd.DataFrame(results)

if df_planb.empty:
    print("No Databricks datasources found. Nothing to write.")
else:
    # Ensure all columns are strings to avoid Spark schema inference conflicts
    for col in df_planb.columns:
        df_planb[col] = df_planb[col].astype(str).fillna("")

    df_planb = df_planb.sort_values(
        by=["gateway_class", "owner_email", "capacity_name", "workspace_name"],
        ascending=[False, True, True, True],
    ).reset_index(drop=True)

    df_personal = df_planb[df_planb["gateway_class"] != "VNet Gateway"].copy()

    # ── Print summary ──
    print("=" * 70)
    print("SUMMARY")
    print("=" * 70)

    print(f"\nBy gateway type:")
    print(df_planb.groupby("gateway_class").size().to_string())

    print(f"\nBy capacity:")
    print(df_planb.groupby(["capacity_name", "capacity_sku", "gateway_class"]).size().to_string())

    if not df_personal.empty:
        print(f"\nBy owner (non-VNet only):")
        print(df_personal.groupby(["owner_email", "gateway_class"]).size().to_string())
        print(f"\nDatasources NOT on VNet: {len(df_personal)}")
        print(f"Unique owners to contact: {df_personal['owner_email'].nunique()}")
    else:
        print("\nAll datasources are on the VNet gateway!")

    # ── Write all connections to Lakehouse Delta table ──
    spark_df_all = spark.createDataFrame(df_planb)
    (
        spark_df_all.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TABLE_ALL)
    )
    print(f"\nWritten {len(df_planb)} rows to Lakehouse table '{TABLE_ALL}'")

    # ── Write personal/non-VNet connections to a separate table ──
    if not df_personal.empty:
        spark_df_personal = spark.createDataFrame(df_personal)
        (
            spark_df_personal.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(TABLE_PERSONAL)
        )
        print(f"Written {len(df_personal)} rows to Lakehouse table '{TABLE_PERSONAL}'")
    else:
        print(f"No personal/non-VNet connections — table '{TABLE_PERSONAL}' not updated.")

    print("\nDone. Tables are now queryable via the Lakehouse SQL endpoint.")

In [ ]:
# Optional: print email templates for dataset owners to contact
# This cell only prints to output and is safe to include in pipeline runs

if not df_planb.empty and not df_personal.empty:
    print("=" * 70)
    print("EMAIL TEMPLATES")
    print("=" * 70)

    for owner_email, grp in df_personal.groupby("owner_email"):
        datasets_list = "\n".join([
            f"   - {row['dataset_name']}"
            f"  (workspace: {row['workspace_name']}"
            f", capacity: {row['capacity_name']})"
            for _, row in grp.iterrows()
        ])
        hosts = grp["host"].unique()
        paths = grp["http_path"].unique()

        print(f"\n{'─' * 60}")
        print(f"To: {owner_email}")
        print(f"Subject: Action Required - Migrate Databricks connection")
        print(f"")
        print(f"Hi {grp.iloc[0]['owner_name']},")
        print(f"")
        print(f"These datasets need migration to VNet Data Gateway:")
        print(f"")
        print(datasets_list)
        print(f"")
        print(f"Host(s): {', '.join(str(h) for h in hosts)}")
        print(f"Path(s): {', '.join(str(p) for p in paths)}")
        print(f"")
        print(f"Steps:")
        print(f"  1. Power BI > Settings > Manage connections and gateways")
        print(f"  2. Select VNet Data Gateway")
        print(f"  3. Add data source > Databricks > enter host + HTTP path")
        print(f"  4. Sign in with OAuth2")
        print(f"  5. Dataset settings > Gateway > select VNet datasource")
        print(f"  6. Refresh dataset")
        print(f"{'─' * 60}")
else:
    print("No personal connections to report.")